## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from langchain_ollama import ChatOllama

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [7]:
MODEL = "llama3.2"
DB_NAME = "vector_db"

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [10]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(model=MODEL,temperature=0)

### These LangChain objects implement the method `invoke()`

In [11]:
retriever.invoke("Who is Avery?")

[Document(id='9d10c870-66f0-4330-9b0f-f4f560a2323d', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [12]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery, so it\'s possible that you\'re referring to one of the following:\n\n1. Avery Brooks: An American actor known for his roles in TV shows such as "Star Trek: Deep Space Nine" and "Designated Survivor".\n2. Avery Whitted: An American professional poker player who has won several tournaments, including the World Series of Poker (WSOP) Main Event.\n3. Avery Wilkerson: An American football linebacker who currently plays for the New York Giants in the National Football League (NFL).\n4. Avery Hunt: An American football quarterback who played college football at the University of Arkansas and was drafted by the Kansas City Chiefs in the 2020 NFL Draft.\n\nIf you could provide more context or information about the Avery you\'re referring to, I\'d be happy to try and provide a more specific answer.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-05-20T11:41:49.7926106Z', 'done': True, 'done_

## Time to put this together!

In [13]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [14]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [15]:
answer_question("Who is Averi Lancaster?", [])

"Avery Lancaster is actually Emily Carter's colleague and not a public figure I have information on. However, based on the context provided, it seems that Avery Lancaster is likely an employee or colleague of Emily Carter at Insurellm.\n\nIf you're looking for more information about Emily Carter, she is a talented individual who exemplifies the kind of talent that drives Insurellm's success and is an invaluable asset to the company."

In [16]:
gr.ChatInterface(answer_question).launch()

c:\Users\Pratham\AIProjects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
